In [ ]:
# Lab 10, notebook 0: bring the lakehouse's SQL view up to date before the quality check reads it.
# Attached lakehouse: freshcart_lake. It runs first in the pipeline, before the quality_tickets dataflow.

### Bring the SQL view up to date

The copy job writes new tickets into the lakehouse's table files. The quality check (the quality_tickets dataflow) reads the same table through the lakehouse's SQL view, which Fabric updates from the files on its own, some time later. Until then the dataflow sees the old table, misses the new day, and still says Succeeded. This cell asks Fabric to update the view now and waits until it is done (about 20 seconds in our tests). It stops with an error if any table was not updated, or if the update takes more than 10 minutes.

In [ ]:
import time, requests

context = notebookutils.runtime.context
workspace, lakehouse = context["currentWorkspaceId"], context["defaultLakehouseId"]
api = "https://api.fabric.microsoft.com/v1"
headers = {"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi")}

lake = requests.get(f"{api}/workspaces/{workspace}/lakehouses/{lakehouse}", headers=headers, timeout=60).json()
sql_view = lake["properties"]["sqlEndpointProperties"]["id"]
answer = requests.post(f"{api}/workspaces/{workspace}/sqlEndpoints/{sql_view}/refreshMetadata",
                       headers=headers, json={}, timeout=60)
answer.raise_for_status()
if answer.status_code == 202:                      # still working: ask again until it is done, at most 10 minutes
    status_url, deadline = answer.headers["Location"], time.time() + 600
    while True:
        time.sleep(int(answer.headers.get("Retry-After", 5)))
        answer = requests.get(status_url, headers=headers, timeout=60)
        answer.raise_for_status()
        state = answer.json().get("status")
        if state in ("Succeeded", "Failed", "Cancelled"):
            break
        if time.time() > deadline:
            raise RuntimeError("The SQL view was still updating after 10 minutes")
    if state != "Succeeded":
        raise RuntimeError(f"The SQL view was not updated: {answer.json()}")
    answer = requests.get(answer.headers.get("Location", status_url + "/result"), headers=headers, timeout=60)
    answer.raise_for_status()
tables = {row["tableName"]: row["status"] for row in answer.json().get("value", [])}
failed = {name: status for name, status in tables.items() if status not in ("Success", "NotRun")}
if failed:
    raise RuntimeError(f"These tables were not updated in the SQL view: {failed}")
print("SQL view up to date. Tables updated now:", [name for name, status in tables.items() if status == "Success"] or "none needed")